# 03 - Graph Construction

Goal: convert the preprocessed traffic flows into graph-ready arrays.

Graph design:

- Node: IP address
- Edge: one traffic flow from `id.orig_h` to `id.resp_h`
- Edge label: `binary_label`
- Node features: simple IP-based features
- Edge features: scaled numeric traffic features plus one-hot categorical traffic features

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.build_graph_arrays import build_graph_arrays  # noqa: E402

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed dir exists:", PROCESSED_DIR.exists())

Project root: c:\Users\Binh\OneDrive\Documents\CẦN NỘP\IDS_GAT_IOT23
Processed dir exists: True


## Build graph arrays

The edge preprocessor is fit only on the training split, then applied to train, validation, and test rows.

In [2]:
metadata = build_graph_arrays(
    processed_dir=PROCESSED_DIR,
    output_dir=PROCESSED_DIR,
)

metadata

{'task': 'binary_edge_classification',
 'graph_design': {'node': 'IP address',
  'edge': 'traffic flow from id.orig_h to id.resp_h',
  'label': 'binary_label'},
 'num_nodes': 92413,
 'num_edges': 100000,
 'node_feature_dim': 8,
 'edge_feature_dim': 55,
 'node_feature_names': ['ipv4_octet_1',
  'ipv4_octet_2',
  'ipv4_octet_3',
  'ipv4_octet_4',
  'is_private',
  'is_global',
  'is_multicast',
  'is_loopback'],
 'edge_feature_names': ['numeric__ts',
  'numeric__id.orig_p',
  'numeric__id.resp_p',
  'numeric__duration',
  'numeric__orig_bytes',
  'numeric__resp_bytes',
  'numeric__missed_bytes',
  'numeric__orig_pkts',
  'numeric__orig_ip_bytes',
  'numeric__resp_pkts',
  'numeric__resp_ip_bytes',
  'categorical__proto_icmp',
  'categorical__proto_tcp',
  'categorical__proto_udp',
  'categorical__service_dhcp',
  'categorical__service_dns',
  'categorical__service_http',
  'categorical__service_irc',
  'categorical__service_missing',
  'categorical__service_ssl',
  'categorical__conn_sta

## Inspect saved arrays

These arrays can be converted to a PyTorch Geometric `Data` object in the next notebook.

In [3]:
arrays = np.load(PROCESSED_DIR / "graph_arrays.npz")

for key in arrays.files:
    print(key, arrays[key].shape, arrays[key].dtype)

x (92413, 8) float32
edge_index (2, 100000) int64
edge_attr (100000, 55) float32
y (100000,) int64
train_mask (100000,) bool
val_mask (100000,) bool
test_mask (100000,) bool


In [ ]:
node_mapping = pd.read_csv(PROCESSED_DIR / "node_mapping.csv")
node_mapping.head()

,ip,node_id
0,0.0.0.0,0
1,1.1.125.138,1
2,1.101.155.200,2
3,1.101.229.206,3
4,1.103.229.0,4


: 

## What this means for GAT

In the next notebook, we will create:

```python
Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
```

Then we will train an edge classifier using node embeddings produced by GAT.